In [5]:
import pandas as pd
import numpy as np
import yaml
import os

from MAST_tools.utils.path_utils import PACKAGE_METADATA_DIR
from preproc_paths import ( 
    DEFAULT_SHOTS_STATS_ALL_FILE,
    DEFAULT_SIGNALS_MEAN_STD_TRAIN_FILE,
)

In [6]:
from tokamark.data_split import get_train_test_val_shots
from tokamark.tools.path import DEFAULT_TOKAMARK_DATA_SPLITS_FILE

train_shots_, test_shots_, val_shots_ = get_train_test_val_shots(
    data_splits_file_path= DEFAULT_TOKAMARK_DATA_SPLITS_FILE
)

print("Train shots:", len(train_shots_))
print("Test shots:", len(test_shots_))
print("Validation shots:", len(val_shots_))
print("Total:", len(train_shots_) + len(val_shots_) + len(test_shots_))


Train shots: 8963
Test shots: 1110
Validation shots: 1115
Total: 11188


## Merge data train, val, test

In [7]:
df_all = pd.read_csv(DEFAULT_SHOTS_STATS_ALL_FILE).sort_values(by=["shot_idx", "variable"])

print("Saved shots_stats_all.csv with", len(df_all), "rows")
print(len(df_all['shot_id'].unique()))

df_train = df_all[df_all['shot_id'].isin(train_shots_)].copy().sort_values(by=["shot_idx", "variable"])
df_val   = df_all[df_all['shot_id'].isin(val_shots_)].copy().sort_values(by=["shot_idx", "variable"])
df_test  = df_all[df_all['shot_id'].isin(test_shots_)].copy().sort_values(by=["shot_idx", "variable"])

print("Train shots:", len(df_train['shot_id'].unique()))
print("Test shots:", len(df_test['shot_id'].unique()))
print("Validation shots:", len(df_val['shot_id'].unique()))
print("Total:", len(df_train['shot_id'].unique()) + len(df_val['shot_id'].unique()) + len(df_test['shot_id'].unique()))

Saved shots_stats_all.csv with 436332 rows
11188
Train shots: 8963
Test shots: 1110
Validation shots: 1115
Total: 11188


## Get train global mean and std for stdscaling

In [5]:
df_train[df_train["shot_id"]==23812].head()

,shot_idx,shot_id,variable,n_dim_shot,n_nans_shot,mean,variance,min,max,median
305682,7838,23812,equilibrium-beta_normal,129.0,39.0,1.736305,0.680695,-0.584141,3.277398,1.600326
305683,7838,23812,equilibrium-beta_pol,129.0,39.0,0.455106,0.046316,-0.092767,0.877834,0.400647
305684,7838,23812,equilibrium-beta_tor,129.0,39.0,3.164488,2.610983,-1.258600,6.050639,2.934588
305685,7838,23812,equilibrium-bphi_rmag,129.0,39.0,-0.535350,0.005481,-0.969970,-0.477782,-0.507632
305686,7838,23812,equilibrium-bvac_rmag,129.0,39.0,-0.486946,0.002470,-0.609947,-0.444663,-0.460203


In [ ]:
# GLOBAL MEAN BASED ON TRAIN
global_mean = (
    df_train
    .groupby("variable")[["n_dim_shot", "n_nans_shot", "mean", ]]
    .apply(lambda g: (g["mean"] * (g["n_dim_shot"] - g["n_nans_shot"])).sum() / (g["n_dim_shot"] - g["n_nans_shot"]).sum())
    .rename("global_mean")
)
df_train_with_group_mean = df_train.join(global_mean, on="variable")

# GLOBAL VARIANCE BASED ON TRAIN
global_variance = (
    df_train_with_group_mean
    .groupby("variable")[["n_dim_shot", "n_nans_shot", "mean", "variance", "global_mean"]]
    .apply(lambda g: ( (g["n_dim_shot"] - g["n_nans_shot"]) * g["variance"] + (g["n_dim_shot"] - g["n_nans_shot"]) * (g["mean"]-g["global_mean"])**2 ).sum() / (g["n_dim_shot"] - g["n_nans_shot"]).sum())
    .rename("global_variance")
)
df_train_with_group_mean_and_variance = df_train_with_group_mean.join(global_variance, on="variable")
df_train_with_group_mean_and_variance.head()

,shot_idx,shot_id,variable,n_dim_shot,n_nans_shot,mean,variance,min,max,median,global_mean,global_variance
31,0,20509,equilibrium-beta_normal,115.0,34.0,1.222396,0.988966,0.050556,3.607922,0.988297,0.937467,1.375627
30,0,20509,equilibrium-beta_pol,115.0,34.0,0.252175,0.037271,0.024701,0.671952,0.214777,0.196528,0.071796
29,0,20509,equilibrium-beta_tor,115.0,34.0,2.930718,6.082609,0.046563,9.005361,2.289963,3.038217,269.323615
33,0,20509,equilibrium-bphi_rmag,115.0,34.0,-0.539910,0.001581,-0.655320,-0.475381,-0.546351,-0.540985,0.004821
32,0,20509,equilibrium-bvac_rmag,115.0,34.0,-0.468917,0.002407,-0.648544,-0.412742,-0.454874,-0.470090,0.003632


#### Remove z-6 outliers from computation

In [23]:
# Z-SCORE COMPUTATION TRAIN
# df_train_with_group_mean_and_variance["z_score"] = (df_train_with_group_mean_and_variance["mean"] - df_train_with_group_mean_and_variance["global_mean"]) / ( np.sqrt(df_train_with_group_mean_and_variance["global_variance"]) / np.sqrt(df_train_with_group_mean_and_variance["n_dim_shot"]) )
df_train_with_group_mean_and_variance["z_score"] = (df_train_with_group_mean_and_variance["mean"] - df_train_with_group_mean_and_variance["global_mean"]) / ( np.sqrt(df_train_with_group_mean_and_variance["global_variance"]) )

df_train_with_group_mean_and_variance

# COUNTING OUTLIERS
print( f'Count of z-6 outliers {sum ( abs(df_train_with_group_mean_and_variance["z_score"]) > 6 )} out of {len(df_train_with_group_mean_and_variance)}')
print( f'Count of z-12 outliers {sum ( abs(df_train_with_group_mean_and_variance["z_score"]) > 12 )} out of {len(df_train_with_group_mean_and_variance)}')

df_train_with_group_mean_and_variance["outlier_z_6"] = abs(df_train_with_group_mean_and_variance["z_score"]) > 6 
df_train_with_group_mean_and_variance["outlier_z_12"] = abs(df_train_with_group_mean_and_variance["z_score"]) > 12

df_train_extended = df_train_with_group_mean_and_variance.copy()
df_train_with_group_mean_and_variance.sort_values('z_score').dropna()

Count of z-6 outliers 37 out of 254475
Count of z-12 outliers 20 out of 254475


,shot_idx,shot_id,variable,n_dim_shot,n_nans_shot,mean,variance,min,max,median,global_mean,global_variance,z_score,outlier_z_6,outlier_z_12
246978,6332,17404,equilibrium-beta_pol,62.0,45.0,-3.030435e+00,1.729355e+01,-1.168474e+01,1.076197e-01,-2.117036e+00,1.965282e-01,7.179563e-02,-12.043284,True,True
246981,6332,17404,equilibrium-bphi_rmag,62.0,45.0,-1.333206e+00,6.021557e-01,-2.879354e+00,-5.473692e-01,-1.202947e+00,-5.409850e-01,4.820974e-03,-11.409830,True,False
246979,6332,17404,equilibrium-beta_normal,62.0,45.0,-9.951116e+00,1.908020e+02,-3.877757e+01,3.311609e-01,-6.845199e+00,9.374668e-01,1.375627e+00,-9.283697,True,False
240975,6178,17417,equilibrium-bphi_rmag,111.0,39.0,-1.184870e+00,2.963725e-01,-2.675583e+00,-6.735463e-01,-8.756120e-01,-5.409850e-01,4.820974e-03,-9.273448,True,False
177012,4538,23767,equilibrium-beta_pol,140.0,122.0,-1.495471e+00,7.830061e-01,-2.933011e+00,-5.297396e-01,-1.250897e+00,1.965282e-01,7.179563e-02,-6.314676,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23117,592,12229,equilibrium-beta_tor,110.0,36.0,2.753833e+02,3.643097e+04,1.804894e+01,5.617272e+02,2.129606e+02,3.038217e+00,2.693236e+02,16.595196,True,True
226385,5804,12223,equilibrium-beta_tor,122.0,33.0,2.865881e+02,3.673864e+04,-1.621598e+01,5.552874e+02,2.507517e+02,3.038217e+00,2.693236e+02,17.277952,True,True
153611,3938,12232,equilibrium-beta_tor,124.0,38.0,3.167945e+02,4.333140e+04,1.421999e+01,6.321520e+02,2.939886e+02,3.038217e+00,2.693236e+02,19.118559,True,True
189305,4853,20038,thomson_scattering-n_e,7200.0,5003.0,1.373060e+21,5.138213e+42,1.192202e+17,1.136142e+22,3.424352e+19,2.248435e+19,3.586026e+39,22.553402,True,True


In [24]:
# import matplotlib.pyplot as plt

# plt.hist(df_train_with_group_mean_and_variance["z_score"], bins=30)
# plt.xlabel("z_score")
# plt.ylabel("Frequency")
# plt.title("Histogram of z-scores")
# plt.show()

In [25]:
df_train_extended.head()

,shot_idx,shot_id,variable,n_dim_shot,n_nans_shot,mean,variance,min,max,median,global_mean,global_variance,z_score,outlier_z_6,outlier_z_12
31,0,20509,equilibrium-beta_normal,115.0,34.0,1.222396,0.988966,0.050556,3.607922,0.988297,0.937467,1.375627,0.242933,False,False
30,0,20509,equilibrium-beta_pol,115.0,34.0,0.252175,0.037271,0.024701,0.671952,0.214777,0.196528,0.071796,0.207679,False,False
29,0,20509,equilibrium-beta_tor,115.0,34.0,2.930718,6.082609,0.046563,9.005361,2.289963,3.038217,269.323615,-0.006550,False,False
33,0,20509,equilibrium-bphi_rmag,115.0,34.0,-0.539910,0.001581,-0.655320,-0.475381,-0.546351,-0.540985,0.004821,0.015485,False,False
32,0,20509,equilibrium-bvac_rmag,115.0,34.0,-0.468917,0.002407,-0.648544,-0.412742,-0.454874,-0.470090,0.003632,0.019475,False,False


In [ ]:
# OG global mean and variance
global_mean = (
    df_train_extended
    .groupby("variable")[["n_dim_shot", "n_nans_shot", "mean"]]
    .apply(lambda g: (g["mean"] * (g["n_dim_shot"] - g["n_nans_shot"])).sum() / (g["n_dim_shot"] - g["n_nans_shot"]).sum())
    .rename("global_mean")
)
global_variance = (
    df_train_extended
    .groupby("variable")[["n_dim_shot", "n_nans_shot", "mean", "variance", "global_mean"]]
    .apply(lambda g: ( (g["n_dim_shot"] - g["n_nans_shot"]) * g["variance"] + (g["n_dim_shot"] - g["n_nans_shot"]) * (g["mean"]-g["global_mean"])**2 ).sum() / (g["n_dim_shot"] - g["n_nans_shot"]).sum())
    .rename("global_variance")
)

# MEAN AND VARIANCE COMPUTATION WITHOUT OUTLIERS Z-6
# global mean and variance without the 6-z outliers
global_mean_no_z_6 = (
    df_train_extended[~df_train_extended['outlier_z_6']]
    .groupby("variable")[["n_dim_shot", "n_nans_shot", "mean"]]
    .apply(lambda g: (g["mean"] * (g["n_dim_shot"] - g["n_nans_shot"])).sum() / (g["n_dim_shot"] - g["n_nans_shot"]).sum())
    .rename("global_mean_no_z_6")
)

df_train_extended = df_train_extended.join(global_mean_no_z_6, on="variable")
global_variance_no_z_6 = (
    df_train_extended[~df_train_extended['outlier_z_6']]
    .groupby("variable")[["n_dim_shot", "n_nans_shot", "mean", "variance", "global_mean_no_z_6"]]
    .apply(lambda g: ( (g["n_dim_shot"] - g["n_nans_shot"]) * g["variance"] + (g["n_dim_shot"] - g["n_nans_shot"]) * (g["mean"]-g["global_mean_no_z_6"])**2 ).sum() / (g["n_dim_shot"] - g["n_nans_shot"]).sum())
    .rename("global_variance_no_z_6")
)


In [ ]:
# Merge all the stats into a single DataFrame
df_stats = pd.DataFrame({
    "variable": global_mean.index,
    # "mean_all": global_mean.values,
    # "std_all": np.sqrt(global_variance.values),
    "mean_no_outliers_z6": global_mean_no_z_6.values,
    "std_no_outliers_z6": np.sqrt(global_variance_no_z_6.values),
})

# Build the dictionary for YAML
final_dict = {}
for _, row in df_stats.iterrows():
    var = row["variable"]
    final_dict[var] = {
        "mean": {
            # "all": row["mean_all"],
            "no_outliers_z6": row["mean_no_outliers_z6"],
            # "no_outliers_z12": row["mean_no_outliers_z12"]
        },
        "std": {
            # "all": row["std_all"],
            "no_outliers_z6": row["std_no_outliers_z6"],
            # "no_outliers_z12": row["std_no_outliers_z12"]
        }
    }

# Write to YAML
with open(DEFAULT_SIGNALS_MEAN_STD_TRAIN_FILE, "w") as f:
    yaml.dump(final_dict, f, sort_keys=False)


## Remove outliers from train, val, test

In [28]:
df_all_with_stats = df_all.join(
    other=global_mean_no_z_6,
    on="variable",
    how="left",
    validate=None
).join(global_variance_no_z_6, on="variable", how="left", validate=None)
df_all_with_stats

df_all_with_stats["z_score"] = (df_all_with_stats["mean"] - df_all_with_stats["global_mean_no_z_6"]) / ( np.sqrt(df_all_with_stats["global_variance_no_z_6"]) )

df_all_with_stats["outlier_z_12"] = abs(df_all_with_stats["z_score"]) > 12

print( f'Count of z-12 outliers {sum ( df_all_with_stats["outlier_z_12"] ) } out of {len(df_all_with_stats)}')
print( f'Spanning {len( df_all_with_stats[ df_all_with_stats["outlier_z_12"]==True ]["variable"].unique() )} variables {df_all_with_stats[ df_all_with_stats["outlier_z_12"]==True ]["variable"].unique() } (out of {len( df_all_with_stats["variable"].unique()) })' )
print( f'Spanning {len( df_all_with_stats[ df_all_with_stats["outlier_z_12"]==True ]["shot_id"].unique() )} shots {df_all_with_stats[ df_all_with_stats["outlier_z_12"]==True ]["shot_id"].unique()} (out of {len( df_all_with_stats["shot_id"].unique()) })' )


Count of z-12 outliers 45 out of 436332
Spanning 8 variables ['equilibrium-beta_tor' 'equilibrium-x_point_r' 'pf_active-coil_voltage'
 'soft_x_rays-horizontal_cam_upper' 'thomson_scattering-n_e'
 'equilibrium-beta_normal' 'equilibrium-beta_pol' 'equilibrium-bphi_rmag'] (out of 39)
Spanning 43 shots [12214 15741 25456 13298 25454 12224 12210 20111 12221 12196 25455 15926
 19461 20166 25453 12232 12216 12219 12209 19953 12230 12218 20038 12213
 20271 19454 12223 19453 12231 17404 12220 12215 12237 12208 12204 19460
 20152 20124 19450 12229 25457 12203 19455] (out of 11188)


In [29]:
# import matplotlib.pyplot as plt

# plt.hist(
#     df_all_with_stats[df_all_with_stats["outlier_z_12"]]["shot_id"],
#     bins=100,
#     color="salmon",
#     edgecolor="black"
# )
# plt.xlabel("shot_id")
# plt.ylabel("Number of outliers")
# plt.show()


In [30]:
zscore_outlier = (
    df_all_with_stats[df_all_with_stats["outlier_z_12"]].groupby("shot_id")["variable"]
    .apply(list)
    .to_dict()
)
print(zscore_outlier)

# with open("dict_zscore_outlier.yaml", "w") as f_:
#     yaml.dump(zscore_outlier, f_, sort_keys=False)
with open("temporal_dict_zscore_outlier.yaml", "w") as f_:
    yaml.dump(zscore_outlier, f_, sort_keys=False)

{12196: ['equilibrium-beta_tor'], 12203: ['equilibrium-beta_tor'], 12204: ['equilibrium-beta_tor'], 12208: ['equilibrium-beta_tor'], 12209: ['equilibrium-beta_tor'], 12210: ['equilibrium-beta_tor'], 12213: ['equilibrium-beta_tor'], 12214: ['equilibrium-beta_tor'], 12215: ['equilibrium-beta_tor'], 12216: ['equilibrium-beta_tor'], 12218: ['equilibrium-beta_tor'], 12219: ['equilibrium-beta_tor'], 12220: ['equilibrium-beta_tor'], 12221: ['equilibrium-beta_tor'], 12223: ['equilibrium-beta_tor'], 12224: ['equilibrium-beta_tor'], 12229: ['equilibrium-beta_tor'], 12230: ['equilibrium-beta_tor'], 12231: ['equilibrium-beta_tor'], 12232: ['equilibrium-beta_tor'], 12237: ['equilibrium-beta_tor'], 13298: ['soft_x_rays-horizontal_cam_upper'], 15741: ['equilibrium-x_point_r'], 15926: ['equilibrium-beta_normal', 'equilibrium-beta_pol', 'equilibrium-bphi_rmag'], 17404: ['equilibrium-beta_pol'], 19450: ['soft_x_rays-horizontal_cam_upper'], 19453: ['soft_x_rays-horizontal_cam_upper'], 19454: ['soft_x_ray

## Combine manual and zscore outliers

In [31]:
import yaml

with open(os.path.join(PACKAGE_METADATA_DIR, "dict_manual_outlier.yaml"), "r") as f:
    manual_outlier = yaml.safe_load(f)

print(manual_outlier)

{11827: ['thomson_scattering-n_e', 'thomson_scattering-t_e'], 11830: ['thomson_scattering-n_e', 'thomson_scattering-t_e'], 11939: ['thomson_scattering-n_e', 'thomson_scattering-t_e'], 11940: ['thomson_scattering-n_e', 'thomson_scattering-t_e'], 11941: ['thomson_scattering-n_e', 'thomson_scattering-t_e'], 11942: ['thomson_scattering-n_e', 'thomson_scattering-t_e'], 11943: ['thomson_scattering-n_e', 'thomson_scattering-t_e'], 11946: ['thomson_scattering-n_e', 'thomson_scattering-t_e'], 11996: ['thomson_scattering-n_e', 'thomson_scattering-t_e'], 11998: ['thomson_scattering-n_e', 'thomson_scattering-t_e'], 12387: ['thomson_scattering-n_e', 'thomson_scattering-t_e'], 12388: ['thomson_scattering-n_e', 'thomson_scattering-t_e'], 12745: ['soft_x_rays-horizontal_cam_lower', 'soft_x_rays-horizontal_cam_upper'], 13041: ['soft_x_rays-horizontal_cam_lower', 'soft_x_rays-horizontal_cam_upper'], 13042: ['soft_x_rays-horizontal_cam_lower', 'soft_x_rays-horizontal_cam_upper'], 13043: ['soft_x_rays-hor

In [32]:
manual_outlier

{11827: ['thomson_scattering-n_e', 'thomson_scattering-t_e'],
 11830: ['thomson_scattering-n_e', 'thomson_scattering-t_e'],
 11939: ['thomson_scattering-n_e', 'thomson_scattering-t_e'],
 11940: ['thomson_scattering-n_e', 'thomson_scattering-t_e'],
 11941: ['thomson_scattering-n_e', 'thomson_scattering-t_e'],
 11942: ['thomson_scattering-n_e', 'thomson_scattering-t_e'],
 11943: ['thomson_scattering-n_e', 'thomson_scattering-t_e'],
 11946: ['thomson_scattering-n_e', 'thomson_scattering-t_e'],
 11996: ['thomson_scattering-n_e', 'thomson_scattering-t_e'],
 11998: ['thomson_scattering-n_e', 'thomson_scattering-t_e'],
 12387: ['thomson_scattering-n_e', 'thomson_scattering-t_e'],
 12388: ['thomson_scattering-n_e', 'thomson_scattering-t_e'],
 12745: ['soft_x_rays-horizontal_cam_lower',
  'soft_x_rays-horizontal_cam_upper'],
 13041: ['soft_x_rays-horizontal_cam_lower',
  'soft_x_rays-horizontal_cam_upper'],
 13042: ['soft_x_rays-horizontal_cam_lower',
  'soft_x_rays-horizontal_cam_upper'],
 130

In [33]:
from collections import defaultdict

combined = defaultdict(set)

for d in (zscore_outlier, manual_outlier):
    for k, v in d.items():
        combined[k].update(v)

combined_outlier = {k: list(v) for k, v in combined.items()}

In [34]:
# with open("dict_outlier_metadata.yaml", "w") as f_:
#     yaml.dump(combined_outlier, f_, sort_keys=False)
with open("temporal_dict_outlier_metadata.yaml", "w") as f_:
    yaml.dump(combined_outlier, f_, sort_keys=False)